In [1]:
import numpy as np
import pandas as pd
import pickle
import h5py
import os

In [2]:
### LOADING ECFP4 FINGERPRINTS FROM HDF5 FILE ###

In [3]:
root = "."
H5_PATH  = os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL_ECFP4.h5")

with h5py.File(H5_PATH, "r") as f:
    fps = f['fps'][:]
    ids = f['ids'][:]
    popc = f['popc'][:]

print("fps shape:", fps.shape, fps.dtype)
print("ids shape:", ids.shape)
print("popc shape:", popc.shape, popc.dtype)

def tanimoto_u64(a: np.ndarray, b: np.ndarray, pa: int, pb: int) -> float:
    inter = sum(int((x & y).bit_count()) for x, y in zip(a, b))
    denom = pa + pb - inter
    return inter / denom if denom else 1.0

def bitbound_possible(pa: int, pb: int, thr: float) -> bool:
    return min(pa, pb) / max(pa, pb) >= thr

fps shape: (9557694, 32) uint64
ids shape: (9557694,)
popc shape: (9557694,) uint16


In [4]:
### LOADING SUCCESS MOLECULES (IN CONFORMATION GENERATION) ###
PATH_TO_INFERRED_PROBS = os.path.join(root, "..", "processed", "unidock_docking", "inference_probs")
success_mols = np.array(pickle.load(open(os.path.join(PATH_TO_INFERRED_PROBS, "success_mols.pkl"), "rb")))

In [10]:
### LOAD MAPPINGS ###
df = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
df['index'] = df.index
id_to_index = dict(zip(df['id'], df['index']))
id_index = pd.Series(df['index'].values, index=df['id'])

In [6]:
### Load pocket detection data ###
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

In [9]:
for file, pocket_n in zip(pocket_detection_data['File name'], pocket_detection_data['Pocket number']):

    # Pocket label
    pocket = file.replace(".pdb", "") + f"_pocket_{pocket_n}"
    
    # Load inference probabilities
    probs = np.load(os.path.join(PATH_TO_INFERRED_PROBS, f"{pocket}_bin_01.npz"))['arr_0']
    inds = np.argsort(probs)[::-1]

    # Sort molecules by inferred probability
    sorted_molecules = success_mols[inds]

    # # From sorted molecules, get their indices in the full dataset
    # sorted_indices = [id_to_index[mol_id] for mol_id in sorted_molecules]

    break